# 02 - Filtragem Colaborativa com KNN

## Objetivo

Implementar e treinar o modelo KNN para filtragem colaborativa, com fundamentação matemática e justificativa da abordagem (user-based vs item-based).

## 1. Setup

In [40]:
import pandas as pd
import numpy as np

from pathlib import Path

from surprise import Dataset, Reader, KNNWithMeans, accuracy, dump
from surprise.model_selection import train_test_split

## 2. Carregamento do dataset tratado

In [41]:
dataset_path = Path("../dataset/dataset.csv")

df = pd.read_csv(dataset_path)

df.head()

,user_id,item_id,ranking,valor_medio_ingresso,tipo_evento
0,1,UM MALLANDRO NA QUARENTENA COM SERGIO MALLANDRO,1,120.0,Show Stand-Up
1,2,DOMINGUINHOS - ISSO AQUI TÁ BOM DEMAIS,1,50.0,Show Musical
2,3,A NOVIÇA REBELDE,1,225.0,Espetáculo Musical
3,4,MUSICAL HAIRSPRAY,1,350.0,Espetáculo Musical
4,5,A NOVIÇA REBELDE,1,125.0,Espetáculo Musical


In [42]:
print("Dimensão do dataset:", df.shape)
print("Usuários únicos:", df["user_id"].nunique())
print("Itens únicos:", df["item_id"].nunique())
print("Ranking máximo:", df["ranking"].max())
print("Ranking mínimo:", df["ranking"].min())

Dimensão do dataset: (109916, 5)
Usuários únicos: 68499
Itens únicos: 449
Ranking máximo: 5
Ranking mínimo: 1


O dataset utilizado neste notebook foi gerado na etapa anterior de análise e preparação dos dados. Ele contém as colunas `user_id`, `item_id` e `ranking`, que representam respectivamente o usuário, o espetáculo e a força da interação entre eles.

Como a base original não possui avaliações explícitas, o `ranking` foi construído como feedback implícito, representando a quantidade de dias distintos em que o usuário interagiu com determinado espetáculo, limitado ao valor máximo de 5.

## 3. Fundamentação Teórica

### 3.1. Lógica do algoritmo

O KNN, ou *K-Nearest Neighbors*, é um algoritmo baseado em proximidade. Em sistemas de recomendação, ele é usado para encontrar usuários ou itens semelhantes a partir de uma matriz de interações.

No contexto deste projeto, a matriz usuário-item representa as interações entre usuários e espetáculos da Aventura. Cada linha representa um usuário, cada coluna representa um espetáculo e cada célula contém o valor de `ranking`, que indica a força da interação entre aquele usuário e aquele espetáculo.

A filtragem colaborativa com KNN funciona em três etapas principais:

1. **Cálculo de similaridade:** o algoritmo mede o quão parecidos são dois usuários ou dois itens com base nos padrões de interação.
2. **Seleção dos k vizinhos mais próximos:** após calcular as similaridades, são selecionados os `k` usuários ou itens mais semelhantes.
3. **Geração da recomendação:** o modelo utiliza os vizinhos mais próximos para estimar quais itens podem ser relevantes para o usuário.

Neste notebook, será utilizado o algoritmo `KNNWithMeans` da biblioteca `scikit-surprise`. Esse algoritmo aplica a lógica do KNN considerando a média das interações, permitindo prever o ranking estimado para pares usuário-item.

Essa abordagem é adequada para o artefato porque permite avaliar o modelo com métricas de regressão, como RMSE e MAE, além de possibilitar a geração de recomendações a partir das predições de ranking.

---

### 3.2. Métricas de similaridade

Para que o KNN consiga encontrar os vizinhos mais próximos, é necessário definir uma métrica de similaridade. Essa métrica determina como o modelo calcula a proximidade entre dois usuários ou dois itens.

As principais métricas utilizadas em sistemas de recomendação são:

#### Similaridade do Cosseno

A similaridade do cosseno mede o ângulo entre dois vetores. No caso de recomendação item-based, cada item é representado por um vetor contendo as interações dos usuários com aquele item.

A fórmula é:

$$
\text{sim}(A,B) = \frac{\vec{A} \cdot \vec{B}}{||\vec{A}|| \cdot ||\vec{B}||}
$$

Onde:

- $\vec{A}$ representa o vetor de interações do item A;
- $\vec{B}$ representa o vetor de interações do item B;
- $\vec{A} \cdot \vec{B}$ representa o produto escalar entre os dois vetores;
- $||\vec{A}||$ representa a norma, ou magnitude, do vetor A;
- $||\vec{B}||$ representa a norma, ou magnitude, do vetor B.

Quanto mais próximo de 1 for o resultado, mais semelhantes são os itens. Quanto mais próximo de 0, menor é a similaridade entre eles.

Neste projeto, a similaridade do cosseno é adequada porque a matriz usuário-item é esparsa, ou seja, possui muitos valores iguais a 0. A métrica do cosseno permite comparar padrões de interação mesmo quando os usuários interagem com poucos itens do catálogo.

#### Correlação de Pearson

A correlação de Pearson mede a relação linear entre dois vetores, considerando a variação em torno da média. Ela é útil quando se deseja reduzir o impacto de usuários que tendem a avaliar tudo muito alto ou muito baixo.

A fórmula geral é:

$$
\text{sim}(u,v) = \frac{\sum_i (r_{ui} - \bar{r}_u)(r_{vi} - \bar{r}_v)}{\sqrt{\sum_i (r_{ui} - \bar{r}_u)^2} \sqrt{\sum_i (r_{vi} - \bar{r}_v)^2}}
$$

Onde:

- $r_{ui}$ é a interação do usuário $u$ com o item $i$;
- $r_{vi}$ é a interação do usuário $v$ com o item $i$;
- $\bar{r}_u$ é a média das interações do usuário $u$;
- $\bar{r}_v$ é a média das interações do usuário $v$.

Apesar de ser útil em bases com avaliações explícitas, como notas de 1 a 5, ela pode ser menos adequada neste caso, pois o `ranking` representa feedback implícito derivado de compras, e não avaliações subjetivas dadas pelos usuários.

#### MSD — Mean Squared Difference

A métrica MSD, ou diferença quadrática média, calcula a distância entre dois vetores com base nas diferenças entre suas interações.

A ideia é medir o quanto dois usuários ou itens se diferenciam em suas interações. Quanto menor a diferença quadrática média, mais semelhantes eles são.

A fórmula pode ser representada como:

$$
MSD(u,v) = \frac{1}{|I_{uv}|} \sum_{i \in I_{uv}} (r_{ui} - r_{vi})^2
$$

Onde:

- $I_{uv}$ representa o conjunto de itens em comum entre os usuários $u$ e $v$;
- $r_{ui}$ é a interação do usuário $u$ com o item $i$;
- $r_{vi}$ é a interação do usuário $v$ com o item $i$.

Essa métrica é simples, mas pode ser mais sensível à escala dos valores de interação.

---

### 3.3. Fórmula de predição

Em abordagens de KNN para recomendação, a predição de uma interação pode ser calculada a partir da média ponderada das interações dos vizinhos mais próximos.

Uma formulação comum, especialmente em modelos como `KNNWithMeans`, é:

$$
\hat{r}_{ui} = \mu_u + \frac{\sum_{v \in N_k(u)} \text{sim}(u,v) \cdot (r_{vi} - \mu_v)}{\sum_{v \in N_k(u)} |\text{sim}(u,v)|}
$$

Onde:

- $\hat{r}_{ui}$ é a interação prevista do usuário $u$ com o item $i$;
- $\mu_u$ é a média das interações do usuário $u$;
- $N_k(u)$ representa o conjunto dos $k$ vizinhos mais próximos do usuário $u$;
- $\text{sim}(u,v)$ é a similaridade entre o usuário $u$ e o vizinho $v$;
- $r_{vi}$ é a interação real do vizinho $v$ com o item $i$;
- $\mu_v$ é a média das interações do vizinho $v$;
- o denominador $\sum_{v \in N_k(u)} |\text{sim}(u,v)|$ normaliza o cálculo pela soma das similaridades absolutas.

Neste notebook, será utilizado o algoritmo `KNNWithMeans` da biblioteca `scikit-surprise`. Esse algoritmo aplica a lógica do KNN considerando a média das interações e os vizinhos mais próximos, permitindo prever o ranking estimado para pares usuário-item.

Essa abordagem é adequada para o artefato porque permite avaliar o modelo com métricas de regressão, como RMSE e MAE, além de possibilitar a geração de recomendações a partir das predições de ranking.

---

### 3.4. Justificativa da métrica escolhida

A métrica escolhida para este modelo foi a **similaridade do cosseno**.

Essa escolha se justifica porque:

- a matriz usuário-item possui alta esparsidade;
- o objetivo é comparar padrões de interação entre espetáculos;
- o cosseno é adequado para comparar vetores mesmo quando há muitos valores 0;
- a escala do `ranking` já foi limitada entre 1 e 5, reduzindo distorções;
- a abordagem facilita a identificação de itens semelhantes com base nos usuários que interagiram com eles.

Portanto, a similaridade do cosseno é uma escolha coerente para o primeiro modelo KNN de filtragem colaborativa aplicado ao sistema de recomendação da Aventura.

## 4. User-based vs Item-based

A filtragem colaborativa com KNN pode ser implementada de duas formas principais: baseada em usuários ou baseada em itens.

Na abordagem **user-based**, o modelo compara usuários entre si. A recomendação é feita a partir dos itens consumidos por usuários parecidos com o usuário alvo.

Na abordagem **item-based**, o modelo compara itens entre si. A recomendação é feita a partir de itens semelhantes àqueles que o usuário já consumiu.

Para este projeto, foi escolhida a abordagem **item-based**.

Essa decisão foi tomada porque o dataset possui uma quantidade muito maior de usuários do que de itens. Na etapa de preparação dos dados, a matriz usuário-item apresentou aproximadamente 68 mil usuários e 571 espetáculos. Dessa forma, é mais eficiente comparar os espetáculos entre si do que comparar todos os usuários.

Além disso, a abordagem item-based é mais interpretável para o contexto da Aventura, pois permite recomendar espetáculos semelhantes aos que o usuário já consumiu.

No `scikit-surprise`, a escolha entre abordagem baseada em usuários ou itens é definida por meio do parâmetro `user_based`.

Neste projeto, foi utilizada a abordagem item-based, configurada com `user_based=False`. Dessa forma, o modelo calcula similaridades entre espetáculos e utiliza esses padrões para prever o interesse dos usuários por itens ainda não consumidos.

## 5. Preparação dos dados para o Surprise

A biblioteca `scikit-surprise` recebe os dados no formato usuário, item e rating. Como a base tratada possui as colunas `user_id`, `item_id` e `ranking`, ela pode ser utilizada diretamente para a construção do modelo.

O `ranking` representa o feedback implícito do usuário com o espetáculo, construído a partir da quantidade de dias distintos de interação e limitado ao valor máximo de 5.

In [43]:
df_surprise = df[["user_id", "item_id", "ranking"]].copy()

reader = Reader(rating_scale=(1, 5))

data = Dataset.load_from_df(
    df_surprise[["user_id", "item_id", "ranking"]],
    reader
)

trainset, testset = train_test_split(
    data,
    test_size=0.2,
    random_state=42
)

print("Dados preparados para o Surprise.")
print("Tamanho do conjunto de teste:", len(testset))

Dados preparados para o Surprise.
Tamanho do conjunto de teste: 21984


## 6. Treinamento do modelo KNN com Surprise

Nesta etapa, foi utilizado o algoritmo `KNNWithMeans` da biblioteca `scikit-surprise`.

Esse modelo calcula a similaridade entre usuários ou itens e realiza a predição considerando a média dos ratings. Como a abordagem escolhida foi item-based, o parâmetro `user_based` foi definido como `False`.

A métrica escolhida foi a similaridade do cosseno, adequada para comparar padrões de interação em uma base esparsa. O valor de `k = 5` foi definido para considerar os cinco vizinhos mais próximos no cálculo da predição.

In [44]:
sim_options = {
    "name": "cosine",
    "user_based": False
}

modelo_knn = KNNWithMeans(
    k=5,
    sim_options=sim_options,
    verbose=True
)

modelo_knn.fit(trainset)

print("Modelo KNNWithMeans treinado com sucesso.")

Computing the cosine similarity matrix...
Done computing similarity matrix.
Modelo KNNWithMeans treinado com sucesso.


## 7. Avaliação do modelo KNN

Após o treinamento, o modelo foi avaliado no conjunto de teste utilizando RMSE e MAE.

O RMSE mede a raiz do erro quadrático médio, penalizando erros maiores com mais intensidade. Já o MAE mede o erro absoluto médio entre o ranking real e o ranking previsto.

Essas métricas permitem avaliar o desempenho do modelo como uma tarefa de predição de ranking.

In [45]:
predictions_knn = modelo_knn.test(testset)

rmse_knn = accuracy.rmse(predictions_knn)
mae_knn = accuracy.mae(predictions_knn)

RMSE: 0.3402
MAE:  0.1366


In [46]:
resultados_knn = pd.DataFrame({
    "modelo": ["KNNWithMeans item-based"],
    "RMSE": [rmse_knn],
    "MAE": [mae_knn]
})

resultados_knn

,modelo,RMSE,MAE
0,KNNWithMeans item-based,0.340209,0.136632


### Interpretação das métricas

O modelo KNN item-based com `KNNWithMeans` obteve RMSE de **0.3402** e MAE de **0.1366** no conjunto de teste.

O RMSE indica a raiz do erro quadrático médio entre o ranking real e o ranking previsto. Como essa métrica penaliza erros maiores, o valor obtido sugere que o modelo apresenta baixo desvio em relação aos rankings observados.

O MAE representa o erro médio absoluto entre as previsões e os valores reais. O valor de 0.1366 indica que, em média, as previsões do modelo se afastam pouco dos rankings reais.

Esses resultados mostram que o KNN conseguiu prever os rankings com boa proximidade dentro da escala de 1 a 5. A comparação final, porém, ainda deverá ser feita com o modelo SVD/Funk SVD para decidir qual abordagem apresenta melhor desempenho.

## 8. Função de recomendação para usuário

A função de recomendação utiliza o modelo treinado para prever o ranking estimado de itens ainda não consumidos por um usuário.

A lógica da função é:

1. identificar os itens já consumidos pelo usuário;
2. listar os itens ainda não consumidos;
3. prever o ranking estimado para cada item não consumido;
4. ordenar os itens pelo maior ranking estimado;
5. retornar os principais itens recomendados.

In [52]:
def recomendar_para_usuario_surprise(modelo, df, user_id, top_n=5):
    """
    Gera recomendações para um usuário usando um modelo KNN treinado com scikit-surprise.

    A função identifica os itens já consumidos pelo usuário, prevê o ranking
    estimado para os itens ainda não consumidos e retorna os itens com maior
    potencial de interesse.

    Parâmetros:
        modelo: modelo treinado do scikit-surprise.
        df: DataFrame contendo as colunas user_id, item_id e ranking.
        user_id: identificador do usuário alvo.
        top_n: quantidade de recomendações desejadas.

    Retorno:
        DataFrame com os itens recomendados e seus rankings estimados.
    """
    if user_id not in df["user_id"].unique():
        raise ValueError(f"O usuário '{user_id}' não foi encontrado na base.")

    todos_itens = df["item_id"].unique()

    itens_consumidos = df[df["user_id"] == user_id]["item_id"].unique()

    itens_nao_consumidos = [
        item for item in todos_itens
        if item not in itens_consumidos
    ]

    predicoes = []

    for item in itens_nao_consumidos:
        pred = modelo.predict(user_id, item)

        predicoes.append({
            "user_id": user_id,
            "item_id": item,
            "ranking_estimado": pred.est
        })

    recomendacoes = (
        pd.DataFrame(predicoes)
        .sort_values("ranking_estimado", ascending=False)
        .head(top_n)
    )

    return recomendacoes

## 9. Teste com usuário de exemplo

Para validar a função de recomendação, foi selecionado um usuário com quantidade moderada de interações na base. Usuários com poucas interações podem gerar recomendações pouco informativas, enquanto usuários extremamente ativos podem representar comportamentos atípicos.

Por isso, foi escolhido um usuário com volume intermediário de consumo, permitindo observar melhor o funcionamento do modelo em um perfil mais representativo.

In [48]:
usuarios_ativos = (
    df.groupby("user_id")
    .size()
    .sort_values(ascending=False)
)

usuarios_validos = usuarios_ativos[
    (usuarios_ativos >= 5) & (usuarios_ativos <= 30)
]

usuario_exemplo = usuarios_validos.index[0]

print("Usuário selecionado:", usuario_exemplo)
print("Quantidade de itens consumidos:", usuarios_validos.loc[usuario_exemplo])

Usuário selecionado: 64907
Quantidade de itens consumidos: 19


In [49]:
print("\nItens já consumidos pelo usuário:")
display(
    df[df["user_id"] == usuario_exemplo][["item_id", "ranking"]]
    .sort_values("ranking", ascending=False)
    .head(10)
)


Itens já consumidos pelo usuário:


,item_id,ranking
104280,NOEL ROSA: COISA NOSSA,3
104269,(1) LONGA JORNADA NOITE ADENTRO,1
104279,MUSICAL HAIRSPRAY,1
104286,UM FILME ARGENTINO - COM LETÍCIA COLIN E MICHE...,1
104285,SRA. KLEIN,1
104284,SINATRA BY KRAMARENCO,1
104283,SILVIO SANTOS VEM AÍ,1
104282,QUANDO EU FOR MÃE QUERO AMAR DESSE JEITO – COM...,1
104281,PRIMA FACIE COM DÉBORA FALABELLA,1
104278,MUSEU NACIONAL [TODAS AS VOZES DO FOGO],1


In [50]:
print("\nRecomendações geradas:")

recomendar_para_usuario_surprise(
    modelo=modelo_knn,
    df=df,
    user_id=usuario_exemplo,
    top_n=5
)


Recomendações geradas:


,user_id,item_id,ranking_estimado
423,64907,PRÊMIO SIM A IGUALDADE RACIAL 2023,4.000000
429,64907,"JOYCE CÂNDIDO EM “UMA SAUDADE IMENSA"" - TRIBUT...",2.000000
354,64907,OS ALACANTOS - FEITIÇO DE ÁQUILA,1.947496
4,64907,FIBRA – FESTIVAL INSTRUMENTAL BRASIL: BEBÊ KRA...,1.691912
58,64907,CAINÃ CAVALCANTE – SINAL DOS TEMPOS / PARTICIP...,1.609992


## 10. Resultados do modelo KNN

Neste notebook, foi implementado um modelo de recomendação com KNN utilizando a biblioteca `scikit-surprise`.

A abordagem escolhida foi item-based, pois a base possui maior quantidade de usuários do que itens, tornando mais viável comparar espetáculos entre si. O modelo foi treinado com `KNNWithMeans`, similaridade do cosseno e `k = 5`.

O modelo foi capaz de prever rankings para pares usuário-item, gerar recomendações para usuários reais e ser avaliado com RMSE e MAE. Essa implementação atende à etapa de filtragem colaborativa com KNN e servirá como base para comparação futura com modelos de fatoração de matrizes, como SVD ou Funk SVD.

## 11. Exportação do modelo

O modelo KNN treinado foi salvo utilizando o método `dump` da biblioteca `scikit-surprise`. Essa etapa permite reutilizar o modelo treinado em etapas futuras de inferência ou comparação com outros modelos.

In [51]:
model_dir = Path("../models")
model_dir.mkdir(parents=True, exist_ok=True)

dump.dump(
    str(model_dir / "knn_surprise_model.joblib"),
    algo=modelo_knn
)

print("Modelo KNN Surprise salvo em:", model_dir)

Modelo KNN Surprise salvo em: ..\models


## 12. Conclusões

Neste notebook, foi implementado um modelo de recomendação utilizando filtragem colaborativa baseada em itens com KNN.

A abordagem item-based foi escolhida porque a base possui uma quantidade muito maior de usuários do que de itens, tornando mais viável comparar espetáculos entre si. Além disso, essa abordagem é mais interpretável para o contexto da Aventura, pois permite recomendar espetáculos semelhantes aos que o usuário já consumiu.

O modelo foi implementado com `KNNWithMeans` da biblioteca `scikit-surprise`, utilizando similaridade do cosseno e `k = 5`. Essa escolha permitiu prever rankings estimados para itens ainda não consumidos e avaliar o desempenho do modelo com RMSE e MAE.

A função de recomendação criada utiliza o modelo treinado para estimar o interesse do usuário em espetáculos ainda não consumidos, retornando os itens com maior ranking previsto.

Como próximos passos, recomenda-se implementar o modelo SVD ou Funk SVD, comparar seus resultados com o KNN e avaliar também métricas de classificação, como precisão, recall, F1-score e acurácia.